In [56]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score, precision_score, recall_score, accuracy_score

In [57]:
df = pd.read_csv("C:/Users/Muhammad Dawood/OneDrive/Documents/Tekkdev/Week 1/clean_tweets.csv")
df

,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,retweet_count,text,tweet_created,tweet_length
0,neutral,1.0000,No reason,0.0000,Virgin America,0,@VirginAmerica What @dhepburn said.,2015-02-24 11:35:00,35
1,positive,0.3486,No reason,0.0000,Virgin America,0,@VirginAmerica plus you've added commercials t...,2015-02-24 11:15:00,72
2,neutral,0.6837,No reason,0.0000,Virgin America,0,@VirginAmerica I didn't today... Must mean I n...,2015-02-24 11:15:00,71
3,negative,1.0000,Bad Flight,0.7033,Virgin America,0,@VirginAmerica it's really aggressive to blast...,2015-02-24 11:15:00,126
4,negative,1.0000,Can't Tell,1.0000,Virgin America,0,@VirginAmerica and it's a really big bad thing...,2015-02-24 11:14:00,55
...,...,...,...,...,...,...,...,...,...
14595,positive,0.3487,No reason,0.0000,American,0,@AmericanAir thank you we got on a different f...,2015-02-22 12:01:00,63
14596,negative,1.0000,Customer Service Issue,1.0000,American,0,@AmericanAir leaving over 20 minutes Late Flig...,2015-02-22 11:59:00,150
14597,neutral,1.0000,No reason,0.0000,American,0,@AmericanAir Please bring American Airlines to...,2015-02-22 11:59:00,60
14598,negative,1.0000,Customer Service Issue,0.6659,American,0,"@AmericanAir you have my money, you change my ...",2015-02-22 11:59:00,135


In [ ]:
# Keep only the columns needed
df = df[['text', 'airline_sentiment']].copy()

In [59]:
df

,text,airline_sentiment
0,@VirginAmerica What @dhepburn said.,neutral
1,@VirginAmerica plus you've added commercials t...,positive
2,@VirginAmerica I didn't today... Must mean I n...,neutral
3,@VirginAmerica it's really aggressive to blast...,negative
4,@VirginAmerica and it's a really big bad thing...,negative
...,...,...
14595,@AmericanAir thank you we got on a different f...,positive
14596,@AmericanAir leaving over 20 minutes Late Flig...,negative
14597,@AmericanAir Please bring American Airlines to...,neutral
14598,"@AmericanAir you have my money, you change my ...",negative


In [60]:
df['airline_sentiment'].unique()    

<ArrowStringArray>
['neutral', 'positive', 'negative']
Length: 3, dtype: str

In [61]:
# Simple stop words list
stop_words = {
    "a", "an", "and", "are", "as", "at", "be", "but", "by", "for", "from",
    "has", "have", "he", "her", "his", "i", "in", "is", "it", "its", "me",
    "my", "of", "on", "or", "our", "so", "that", "the", "their", "them",
    "there", "they", "this", "to", "was", "were", "what", "when", "where",
    "which", "who", "will", "with", "you", "your", "about", "can", "could",
    "would", "should", "dont", "didnt", "im", "ive", "we", "had", "did"
}


In [62]:
# Simple slang normalization
slang_map = {
    "luv": "love",
    "plz": "please",
    "pls": "please",
    "wth": "what the hell",
    "omg": "oh my god",
    "cant": "cannot",
    "can't": "cannot",
    "wanna": "want to",
    "gonna": "going to",
    "b4": "before",
    "u": "you",
    "r": "are",
    "n't": " not"
}

In [63]:
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)   # remove URLs
    text = re.sub(r"@\w+", " ", text)               # remove mentions
    text = re.sub(r"#[\w-]+", " ", text)            # remove hashtags
    text = re.sub(r"[^a-z0-9\s]", " ", text)        # remove emojis/punctuation
    text = re.sub(r"\s+", " ", text).strip()

    words = text.split()
    words = [slang_map.get(w, w) for w in words]
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

In [64]:
# Apply preprocessing
df["clean_text"] = df["text"].apply(preprocess_text)

In [65]:
df.isnull().sum()

text                 0
airline_sentiment    0
clean_text           0
dtype: int64

In [66]:
df

,text,airline_sentiment,clean_text
0,@VirginAmerica What @dhepburn said.,neutral,said
1,@VirginAmerica plus you've added commercials t...,positive,plus ve added commercials experience tacky
2,@VirginAmerica I didn't today... Must mean I n...,neutral,didn t today must mean need take another trip
3,@VirginAmerica it's really aggressive to blast...,negative,s really aggressive blast obnoxious entertainm...
4,@VirginAmerica and it's a really big bad thing...,negative,s really big bad thing
...,...,...,...
14595,@AmericanAir thank you we got on a different f...,positive,thank got different flight chicago
14596,@AmericanAir leaving over 20 minutes Late Flig...,negative,leaving over 20 minutes late flight no warning...
14597,@AmericanAir Please bring American Airlines to...,neutral,please bring american airlines
14598,"@AmericanAir you have my money, you change my ...",negative,money change flight don t answer phones any ot...


In [67]:
X = df["clean_text"]
y = df["airline_sentiment"]

In [68]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [69]:
# Baseline model: TF-IDF + Logistic Regression
pipeline = make_pipeline(
    TfidfVectorizer(ngram_range=(1, 2), min_df=2),
    LogisticRegression(C=2, max_iter=1000, class_weight="balanced")
)

In [70]:
pipeline.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](3,)","['negative','neutral','positive']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [71]:
y_pred = pipeline.predict(X_test)
y_pred

array(['neutral', 'negative', 'neutral', ..., 'positive', 'negative',
       'negative'], shape=(2920,), dtype=object)

In [ ]:
# Evaluation
print(classification_report(y_test, y_pred))
print("F1 Macro:", round(f1_score(y_test, y_pred, average="macro"), 4))
print("Precision Macro:", round(precision_score(y_test, y_pred, average="macro", zero_division=0), 4))
print("Recall Macro:", round(recall_score(y_test, y_pred, average="macro", zero_division=0), 4))

              precision    recall  f1-score   support

    negative       0.89      0.84      0.86      1831
     neutral       0.59      0.67      0.63       618
    positive       0.70      0.72      0.71       471

    accuracy                           0.79      2920
   macro avg       0.73      0.74      0.73      2920
weighted avg       0.79      0.79      0.79      2920

F1 Macro: 0.7346
Precision Macro: 0.7265
Recall Macro: 0.745


In [ ]:
new_tweet = "He is very nice!"
#He is a prespecious man!
cleaned_tweet = preprocess_text(new_tweet)
prediction = pipeline.predict([cleaned_tweet])[0]

print("Input:", new_tweet)
print("Cleaned:", cleaned_tweet)
print("Predicted sentiment:", prediction)

Input: He is a prespecious man!
Cleaned: prespecious man
Predicted sentiment: neutral
